# 3. Supertonic 3 — warsztat zaawansowany i serwer

Notebook rozszerza podstawowy przykład o parametry jakości, szybkości, dzielenia długiego tekstu, opcjonalny własny styl głosu oraz uruchomienie backendu FastAPI dla istniejącego HTML.

> Serwer jest uruchamiany dopiero w wyraźnie oznaczonej komórce. Można go zatrzymać w ostatniej komórce.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request

from IPython.display import Audio, display
from supertonic import TTS

def find_project_dir() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, current / "302-tts-supertonic", *current.parents]:
        if (candidate / "requirements.txt").is_file() and (candidate / "server" / "main.py").is_file():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu 302-tts-supertonic.")

PROJECT_DIR = find_project_dir()
VENV_DIR = (PROJECT_DIR / ".venv").resolve()
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if Path(sys.prefix).resolve() != VENV_DIR:
    raise RuntimeError("Przełącz kernel na: Supertonic Workshop (.venv).")

print("Projekt:", PROJECT_DIR)

## Sterowanie jakością i tempem

`total_steps` steruje kompromisem jakość/czas (5–12), `speed` tempem (0.7–2.0), a `max_chunk_length` długością fragmentów dłuższego tekstu.

In [ ]:
ADVANCED_REQUEST = {
    "text": (
        "Synteza mowy może pracować z dłuższym tekstem. "
        "Możemy kontrolować jakość, tempo oraz przerwy pomiędzy fragmentami."
    ),
    "voice": "F4",
    "language": "pl",
    "total_steps": 10,
    "speed": 0.95,
    "max_chunk_length": 180,
    "silence_duration": 0.25,
}

(OUTPUT_DIR / "03-advanced-request.json").write_text(
    json.dumps(ADVANCED_REQUEST, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(ADVANCED_REQUEST, ensure_ascii=False, indent=2))

In [ ]:
tts = TTS(auto_download=True)
style = tts.get_voice_style(voice_name=ADVANCED_REQUEST["voice"])
wav, duration = tts.synthesize(
    text=ADVANCED_REQUEST["text"],
    voice_style=style,
    lang=ADVANCED_REQUEST["language"],
    total_steps=ADVANCED_REQUEST["total_steps"],
    speed=ADVANCED_REQUEST["speed"],
    max_chunk_length=ADVANCED_REQUEST["max_chunk_length"],
    silence_duration=ADVANCED_REQUEST["silence_duration"],
)
advanced_wav = OUTPUT_DIR / "03-advanced-output.wav"
tts.save_audio(wav, str(advanced_wav))
print(f"Czas nagrania: {float(duration[0]):.2f} s")
display(Audio(filename=str(advanced_wav)))

## Porównanie wariantów

Dodaj lub usuń konfiguracje. Każdy wariant otrzyma osobny WAV. Oprócz głosu, jakości i tempa przetestuj długość fragmentów, przerwy między nimi oraz tryb `verbose`.

In [ ]:
EXPERIMENT_TEXT = (
    "Czy słyszysz różnicę? To jest drugie zdanie! "
    "Teraz sprawdzamy tempo, jakość oraz długość fragmentów."
)
EXPERIMENTS = [
    {"name": "szybko", "voice": "F4", "steps": 5, "speed": 1.25, "chunk": 180, "silence": 0.15},
    {"name": "dokladnie", "voice": "F4", "steps": 12, "speed": 1.0, "chunk": 180, "silence": 0.3},
    {"name": "inny-glos", "voice": "M3", "steps": 8, "speed": 1.0, "chunk": 180, "silence": 0.3},
    {"name": "krotkie-fragmenty", "voice": "F4", "steps": 8, "speed": 1.0, "chunk": 45, "silence": 0.5},
]

for experiment in EXPERIMENTS:
    experiment_style = tts.get_voice_style(voice_name=experiment["voice"])
    started_at = time.perf_counter()
    experiment_wav, _ = tts.synthesize(
        text=EXPERIMENT_TEXT,
        voice_style=experiment_style,
        lang="pl",
        total_steps=experiment["steps"],
        speed=experiment["speed"],
        max_chunk_length=experiment["chunk"],
        silence_duration=experiment["silence"],
        verbose=True,
    )
    path = OUTPUT_DIR / f"porownanie-{experiment['name']}.wav"
    tts.save_audio(experiment_wav, str(path))
    print(path.name, f"| generowanie: {time.perf_counter() - started_at:.2f} s")
    display(Audio(filename=str(path)))

## Jak Supertonic rozpoznaje granice zdań?

Wersja 1.3.1 rozpoznaje `.`, `?` i `!` jako zakończenia zdań podczas automatycznego chunkowania. Poniższy test pokazuje fragmenty bez uruchamiania kolejnej syntezy. Zmień `max_len` i dodaj przecinki, skróty lub dłuższe zdanie.

In [ ]:
from supertonic.utils import chunk_text

TEXT_TO_SPLIT = (
    "To jest oznajmienie. Czy to jest pytanie? Tak, to jest wykrzyknienie! "
    "Na końcu znajduje się jeszcze jedno zdanie do testu."
)

for index, chunk in enumerate(chunk_text(TEXT_TO_SPLIT, max_len=55), start=1):
    print(f"{index}: {chunk!r} ({len(chunk)} znaków)")

## *Opcjonalny własny styl głosu z JSON (Opcjonalne)

Wstaw ścieżkę do legalnie pozyskanego pliku JSON wyeksportowanego z Voice Builder. Pozostaw `None`, aby pominąć komórkę. Nie używaj głosu innej osoby bez jej zgody.

In [ ]:
CUSTOM_STYLE_PATH = None  # np. PROJECT_DIR / "voices" / "moj-glos.json"

if CUSTOM_STYLE_PATH is None:
    print("Pomijam własny styl głosu.")
else:
    custom_path = Path(CUSTOM_STYLE_PATH).expanduser().resolve()
    custom_style = tts.get_voice_style_from_path(str(custom_path))
    custom_wav, _ = tts.synthesize(
        text="To jest test własnego, świadomie udostępnionego stylu głosu.",
        voice_style=custom_style, lang="pl", total_steps=8,
    )
    custom_output = OUTPUT_DIR / "wlasny-styl.wav"
    tts.save_audio(custom_wav, str(custom_output))
    display(Audio(filename=str(custom_output)))

## Uruchomienie serwera dla HTML

Komórka uruchamia `server/main.py` pod `http://127.0.0.1:8000`. Otwórz ten adres w przeglądarce — serwer udostępni `webgui/index.html` i API z tego samego hosta.

Pierwsze żądanie TTS może potrwać dłużej, ponieważ osobny proces serwera ładuje model.

In [ ]:
SERVER_URL = "http://127.0.0.1:8000"
SERVER_LOG_PATH = PROJECT_DIR / "server" / "server.log"
server_log = SERVER_LOG_PATH.open("w", encoding="utf-8")
SERVER_PROCESS = subprocess.Popen(
    [sys.executable, str(PROJECT_DIR / "server" / "main.py")],
    cwd=PROJECT_DIR, stdout=server_log, stderr=subprocess.STDOUT, text=True,
)

for _ in range(40):
    if SERVER_PROCESS.poll() is not None:
        server_log.close()
        raise RuntimeError(f"Serwer zakończył pracę. Sprawdź {SERVER_LOG_PATH}")
    try:
        with urllib.request.urlopen(f"{SERVER_URL}/health", timeout=1) as response:
            health = json.load(response)
        print("Serwer działa:", health)
        print("Otwórz:", SERVER_URL)
        break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(0.25)
else:
    raise TimeoutError(f"Serwer nie wystartował. Sprawdź {SERVER_LOG_PATH}")

## Test API takim samym JSON-em jak z HTML

Pierwsze żądanie sprawdza podgląd WAV. Drugie tworzy trwały plik w `generated_audio/` i zwraca link pobrania.

In [ ]:
api_payload = json.dumps({
    "text": "To nagranie powstało przez warsztatowe API FastAPI.",
    "voice": "F2",
    "language": "pl",
}).encode("utf-8")

preview_request = urllib.request.Request(
    f"{SERVER_URL}/api/tts/preview", data=api_payload,
    headers={"Content-Type": "application/json"}, method="POST",
)
with urllib.request.urlopen(preview_request, timeout=180) as response:
    preview_bytes = response.read()

api_preview_path = OUTPUT_DIR / "api-preview.wav"
api_preview_path.write_bytes(preview_bytes)
print("Podgląd:", api_preview_path, len(preview_bytes), "bajtów")
display(Audio(filename=str(api_preview_path)))

In [ ]:
file_request = urllib.request.Request(
    f"{SERVER_URL}/api/tts/files", data=api_payload,
    headers={"Content-Type": "application/json"}, method="POST",
)
with urllib.request.urlopen(file_request, timeout=180) as response:
    created_file = json.load(response)

print(json.dumps(created_file, ensure_ascii=False, indent=2))
print("Link:", SERVER_URL + created_file["download_url"])

## Zatrzymanie serwera

Uruchom tę komórkę po zakończeniu ćwiczeń. Notebook 5 nie usuwa procesów automatycznie, dlatego serwer należy zatrzymać tutaj.

In [ ]:
if "SERVER_PROCESS" in globals() and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=10)
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()
        SERVER_PROCESS.wait(timeout=5)

if "server_log" in globals() and not server_log.closed:
    server_log.close()

print("Serwer zatrzymany.")